# SMS Spam Detection using Simple RNN

This clean portfolio notebook demonstrates privacy-safe sample loading, shared
text preprocessing, integer sequence generation, saved Simple RNN inference,
held-out evaluation, baseline comparison, visual diagnostics, and error analysis.

The full corpus is downloaded only during optional retraining.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import clean_sms_frame
from src.spam_detection_pipeline import load_artifacts, predict_messages


## Load privacy-safe sample messages


In [ ]:
sample = pd.read_csv(
    PROJECT_ROOT / "data" / "sample_sms_messages.csv"
)
sample.head()


## Validate and clean messages


In [ ]:
cleaned_sample, quality_report = clean_sms_frame(
    sample,
    message_column="message",
)
quality_report


## Load the saved Simple RNN and tokenizer


In [ ]:
artifacts = load_artifacts()
artifacts.model_metadata


## Generate predictions


In [ ]:
predictions, reports = predict_messages(
    cleaned_sample["message"].tolist(),
    artifacts,
)
predictions[[
    "message",
    "predicted_class",
    "spam_probability",
    "decision_confidence",
    "confidence_band",
]].head(10)


## Held-out results and baselines


In [ ]:
metrics = json.loads(
    (PROJECT_ROOT / "outputs" / "model_metrics.json").read_text()
)
comparison = pd.read_csv(
    PROJECT_ROOT / "outputs" / "model_comparison.csv"
)
metrics, comparison


## Evaluation visuals


In [ ]:
from IPython.display import Image, display

for filename in [
    "class_distribution.png",
    "message_length_distribution.png",
    "confusion_matrix.png",
    "roc_curve.png",
    "precision_recall_curve.png",
    "threshold_analysis.png",
    "training_accuracy.png",
    "training_loss.png",
    "model_comparison.png",
]:
    display(Image(filename=str(PROJECT_ROOT / "outputs" / filename)))


## Error analysis


In [ ]:
errors = pd.read_csv(
    PROJECT_ROOT / "outputs" / "error_analysis.csv"
)
errors.head(15)


## Conclusion

The improved Simple RNN substantially outperforms the original notebook result
after duplicate removal, leak-free splitting, class weighting, early stopping,
and validation-only threshold selection. TF-IDF Logistic Regression remains the
strongest test model, showing that complexity does not guarantee superiority.
